# 10b. 베이스라인 학습 및 롤링 추론

**입력**: `10a`에서 생성한 6:1:1:2 분할 데이터  
특징: `ST4000DM000_v3.parquet`의 **원본 변수** (파생 변수 없음)

## 두 가지 베이스라인

| | 방법 A: No-Tricks | 방법 B: UnderBagging |
|---|---|---|
| 학습 데이터 | train (그대로 전체) | train → 10개 서브셋 (10:1, near-fail 3배) |
| 임계값 탐색 | val_calib (FAR 제약) | val_calib (FAR 제약) |
| 최종 평가 | test (고장당일 포함) | test (고장당일 포함) |

## 평가 기준
- **생애 최초 알람**이 리드타임(10/20/30일) 내 → Hit
- Disk FAR: 정상 개체 중 알람 발령 비율
- Disk Recall: 고장 개체 중 리드타임 내 탐지 비율

## 0. 환경 설정

In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, lightgbm as lgb
import joblib, json, time
from pathlib import Path

SPLIT_DIR = Path(r"C:/Workspace/06_ML_projdect/26_1_COIN/data/10_baseline_split")
MODEL_DIR = Path(r"C:/Workspace/06_ML_projdect/26_1_COIN/models/10_baseline")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

META_COLS  = ["serial_number", "date", "failure"]
LEAD_TIMES = [10, 20, 30]

# 공통 LightGBM 파라미터 (단순 베이스라인용, Optuna 미사용)
BASE_PARAMS = {
    "objective"        : "binary",
    "metric"           : "average_precision",
    "learning_rate"    : 0.05,
    "max_depth"        : 6,
    "num_leaves"       : 63,
    "min_child_samples": 50,
    "feature_fraction" : 0.8,
    "n_estimators"     : 500,
    "random_state"     : 42,
    "n_jobs"           : -1,
    "verbosity"        : -1,
}

# 언더배깅 설정
N_SUBSETS      = 10
NORMAL_RATIO   = 10
NEAR_FAIL_DAYS = 30   # D-11 ~ D-30
NEAR_FAIL_W    = 3

print("환경 설정 완료")

환경 설정 완료


## 1. 데이터 로드

In [2]:
print("데이터 로드 중...")
df_train    = pd.read_parquet(SPLIT_DIR / "train_raw.parquet")
df_val_tune = pd.read_parquet(SPLIT_DIR / "val_tune_raw.parquet")
df_val_calib= pd.read_parquet(SPLIT_DIR / "val_calib_raw.parquet")  # 고장당일 포함
df_test     = pd.read_parquet(SPLIT_DIR / "test_raw.parquet")        # 고장당일 포함

FEATURE_COLS = [c for c in df_train.columns if c not in META_COLS]

for name, df in [("train",df_train),("val_tune",df_val_tune),("val_calib",df_val_calib),("test",df_test)]:
    f = df['failure'].sum()
    print(f"  {name:12s}: {len(df):>12,} rows | failure={f:,} ({f/len(df)*100:.4f}%)")

print(f"\n피처 수: {len(FEATURE_COLS)}")
print(f"피처: {FEATURE_COLS}")

데이터 로드 중...


  train       :   47,810,330 rows | failure=33,560 (0.0702%)
  val_tune    :    7,982,747 rows | failure=5,616 (0.0704%)
  val_calib   :    7,975,356 rows | failure=6,201 (0.0778%)
  test        :   15,927,427 rows | failure=12,288 (0.0771%)

피처 수: 24
피처: ['smart_3_raw', 'smart_4_raw', 'smart_5_raw', 'smart_9_raw', 'smart_10_raw', 'smart_183_raw', 'smart_184_raw', 'smart_187_raw', 'smart_189_raw', 'smart_191_raw', 'smart_192_raw', 'smart_193_raw', 'smart_197_raw', 'smart_198_raw', 'smart_199_raw', 'smart_241_raw', 'smart_242_raw', 'total_reads', 'seek_error_count', 'total_seeks', 'timeout_total', 'timeout_5s', 'smart_190_raw', 'smart_194_raw']


## 유틸: 고속 롤링 추론 & Disk-level 평가 함수

In [ ]:
def predict_rolling(models, df, feature_cols):
    """앙상블 평균 확률 예측 (시간 순서 유지)."""
    df = df.copy()
    df["base_serial"] = df["serial_number"].str.replace(r'_\d+$', '', regex=True)
    df_s = df.sort_values(["base_serial", "date"]).reset_index(drop=True)
    X    = df_s[feature_cols].values
    # predict_proba를 사용하여 확률값 산출 (0/1 predict 오용 버그 수정)
    df_s["pred_prob"] = np.mean([m.predict_proba(X)[:, 1] for m in models], axis=0)
    return df_s

def prepare_eval_data(df_pred):
    """디스크 단위 평가 데이터를 미리 정렬 및 그룹화하여 루프 연산 속도 최적화."""
    print("  평가 데이터 그룹화 및 정렬 중...")
    disks = []
    # df_pred is already sorted by ["base_serial", "date"]
    for sn, grp in df_pred.groupby("base_serial", sort=False):
        is_failed = grp["failure"].max() == 1
        dates = grp["date"].values
        probs = grp["pred_prob"].values
        
        fail_date = None
        if is_failed:
            fail_date = grp.loc[grp["failure"] == 1, "date"].max()
            
        disks.append({
            "sn": sn,
            "is_failed": is_failed,
            "dates": dates,
            "probs": probs,
            "fail_date": pd.Timestamp(fail_date) if fail_date is not None else None
        })
    return disks

def disk_level_eval_fast(disks, threshold, lead_times):
    """
    고속 디스크 단위 롤링 평가.
    - 최초 알람 날짜가 실제 고장일(failure=1 중 마지막 날) 기준 lead_time 이내 → Hit
    - 정상 개체: 알람 1회라도 → False Alarm
    """
    records = []
    for d in disks:
        is_failed = d["is_failed"]
        probs = d["probs"]
        dates = d["dates"]
        
        # 임계값 이상의 알람 인덱스 확인
        alarm_idx = np.where(probs >= threshold)[0]
        has_alarm = len(alarm_idx) > 0
        
        if is_failed:
            if has_alarm:
                first_alarm = pd.Timestamp(dates[alarm_idx[0]])
                lead = (d["fail_date"] - first_alarm).days
            else:
                lead = -1  # 미탐
            records.append({"is_failed": True, "lead_days": lead})
        else:
            records.append({"is_failed": False, "false_alarm": 1 if has_alarm else 0})
            
    res = pd.DataFrame(records)
    fail = res[res["is_failed"]]
    norm = res[~res["is_failed"]]
    
    disk_far = norm["false_alarm"].mean() * 100 if len(norm) > 0 else 0.0
    summary = {"threshold": threshold, "Disk_FAR(%)": round(disk_far, 4)}
    for lt in lead_times:
        hits = (fail["lead_days"] >= 0) & (fail["lead_days"] <= lt)
        recall = hits.sum() / len(fail) * 100 if len(fail) > 0 else 0.0
        summary[f"Recall_{lt}d(%)"] = round(recall, 4)
    return summary, res

def threshold_search(disks, lead_time=30, target_far=1.0, n_steps=1000):
    """FAR 제약(<= 1%) 하에 Recall 최대화 임계값 탐색."""
    best = None
    # 0.0001부터 0.9999까지 정밀 탐색
    for thr in np.linspace(0.0001, 0.9999, n_steps):
        s, _ = disk_level_eval_fast(disks, thr, [lead_time])
        if s["Disk_FAR(%)"] <= target_far:
            if best is None or s[f"Recall_{lead_time}d(%)"] > best[f"Recall_{lead_time}d(%)"] or \
               (s[f"Recall_{lead_time}d(%)"] == best[f"Recall_{lead_time}d(%)"] and thr > best["threshold"]):
                best = s
    return best

print("유틸 함수 정의 완료")

---
## 방법 A: 단순 통합 학습 (No-Tricks Baseline)

언더샘플링·앙상블 없이 **train 전체**를 단일 LightGBM으로 학습.

In [4]:
print("[방법 A] train 단일 모델 학습...")
t0 = time.time()

X_tr = df_train[FEATURE_COLS]
y_tr = df_train["failure"]

model_a = lgb.LGBMClassifier(**BASE_PARAMS)
model_a.fit(X_tr, y_tr)

print(f"  완료 ({time.time()-t0:.1f}초)  |  train rows={len(df_train):,}  failure={y_tr.sum():,}")

[방법 A] train 단일 모델 학습...


  완료 (160.6초)  |  train rows=47,810,330  failure=33,560


In [5]:
print("[방법 A] val_calib으로 임계값 탐색 (FAR≤1%, lead=30d)...")
df_calib_pred_a = predict_rolling([model_a], df_val_calib, FEATURE_COLS)

print("  예측 확률 분포 백분위수:")
print(df_calib_pred_a["pred_prob"].describe(percentiles=[0.5, 0.9, 0.99, 0.999]))

calib_disks_a = prepare_eval_data(df_calib_pred_a)
best_a  = threshold_search(calib_disks_a, lead_time=30, target_far=1.0)
thr_a   = best_a["threshold"] if best_a else 0.5
print(f"  최적 임계값: {thr_a:.4f}  (val_calib: {best_a})")

print("\n[방법 A] test 롤링 추론 & 평가...")
df_test_pred_a = predict_rolling([model_a], df_test, FEATURE_COLS)
test_disks_a = prepare_eval_data(df_test_pred_a)

rows_a = []
for thr in sorted({0.01, 0.1, 0.5, 0.9, round(thr_a, 4)}):
    s, _ = disk_level_eval_fast(test_disks_a, thr, LEAD_TIMES)
    rows_a.append(s)

df_res_a = pd.DataFrame(rows_a)
print(df_res_a.to_string(index=False))

[방법 A] val_calib으로 임계값 탐색 (FAR≤1%, lead=30d)...


  예측 확률 분포 백분위수:
count    7.975356e+06
mean     1.826670e-03
std      3.306601e-02
min      0.000000e+00
50%      2.337635e-04
90%      6.212611e-04
99%      7.958843e-03
99.9%    7.871656e-01
max      1.000000e+00
Name: pred_prob, dtype: float64
  평가 데이터 그룹화 및 정렬 중...


  최적 임계값: 0.2623  (val_calib: {'threshold': np.float64(0.26230980980980984), 'Disk_FAR(%)': np.float64(0.9877), 'Recall_30d(%)': np.float64(44.0917)})

[방법 A] test 롤링 추론 & 평가...


  평가 데이터 그룹화 및 정렬 중...


 threshold  Disk_FAR(%)  Recall_10d(%)  Recall_20d(%)  Recall_30d(%)
    0.0100       5.7319        36.0989        42.5419        46.1606
    0.1000       1.5962        41.5711        46.0724        48.2789
    0.2623       1.0702        42.2771        45.1898        46.3372
    0.5000       0.7316        41.3945        43.4245        44.3071
    0.9000       0.5260        38.3936        39.8058        40.6002


---
## 방법 B: 언더배깅 앙상블 (기존 파이프라인 동일 방식)

### 샘플링
- 모든 failure 행 포함
- normal 행: 10:1 비율, Near-failure(D-11~D-30) 3배 가중 비복원 추출
- 10개 서브셋 → 각각 LightGBM → 평균 앙상블

In [ ]:
def build_subsets(df_tr, n_subsets=10, normal_ratio=10,
                  near_fail_days=30, near_fail_w=3, seed=42):
    rng       = np.random.default_rng(seed)
    df_tr     = df_tr.copy()
    df_tr["base_serial"] = df_tr["serial_number"].str.replace(r'_\d+$', '', regex=True)
    
    fail_df   = df_tr[df_tr["failure"] == 1].copy()
    normal_df = df_tr[df_tr["failure"] == 0].copy()
    n_per     = len(fail_df) * normal_ratio

    # near-failure 가중치: 고장 개체의 D-11 ~ D-30 구간 (base_serial 기준)
    fail_dates = (
        df_tr[df_tr["failure"]==1]
        .groupby("base_serial")["date"].max()
        .reset_index().rename(columns={"date":"fail_date"})
    )
    fail_dates["fail_date"] = pd.to_datetime(fail_dates["fail_date"])
    normal_df = normal_df.merge(fail_dates, on="base_serial", how="left")
    normal_df["date"] = pd.to_datetime(normal_df["date"])
    normal_df["dtf"]  = (normal_df["fail_date"] - normal_df["date"]).dt.days
    is_near = (normal_df["dtf"] > 10) & (normal_df["dtf"] <= near_fail_days)
    normal_df["w"] = np.where(is_near, near_fail_w, 1).astype(float)
    normal_df["w"] /= normal_df["w"].sum()

    idx = normal_df.index.tolist()
    w   = normal_df["w"].values

    subsets = []
    for i in range(n_subsets):
        chosen  = rng.choice(idx, size=min(n_per, len(idx)), replace=False, p=w)
        sampled = normal_df.loc[chosen]
        sub     = pd.concat([fail_df, sampled], ignore_index=True)
        subsets.append(sub)
        print(f"  Subset {i+1:02d}: {len(sub):,} (fail={len(fail_df):,}, normal={len(sampled):,})")
    return subsets

print("[방법 B] 언더배깅 서브셋 생성...")
t0      = time.time()
subsets = build_subsets(df_train, N_SUBSETS, NORMAL_RATIO, NEAR_FAIL_DAYS, NEAR_FAIL_W)
print(f"  완료 ({time.time()-t0:.1f}초)")

In [7]:
print("[방법 B] 10개 서브셋 학습...")
t0 = time.time()
models_b = []
for i, sub in enumerate(subsets):
    m = lgb.LGBMClassifier(**BASE_PARAMS)
    m.fit(sub[FEATURE_COLS], sub["failure"])
    models_b.append(m)
    print(f"  Subset {i+1:02d} 완료")
print(f"\n✅ 앙상블 학습 완료 ({time.time()-t0:.1f}초)")

[방법 B] 10개 서브셋 학습...


  Subset 01 완료


  Subset 02 완료


  Subset 03 완료


  Subset 04 완료


...

  Subset 06 완료


  Subset 07 완료


  Subset 08 완료


  Subset 09 완료


  Subset 10 완료

✅ 앙상블 학습 완료 (18.1초)


In [8]:
print("[방법 B] val_calib으로 임계값 탐색 (FAR≤1%, lead=30d)...")
df_calib_pred_b = predict_rolling(models_b, df_val_calib, FEATURE_COLS)

print("  예측 확률 분포 백분위수:")
print(df_calib_pred_b["pred_prob"].describe(percentiles=[0.5, 0.9, 0.99, 0.999]))

calib_disks_b = prepare_eval_data(df_calib_pred_b)
best_b  = threshold_search(calib_disks_b, lead_time=30, target_far=1.0)
thr_b   = best_b["threshold"] if best_b else 0.5
print(f"  최적 임계값: {thr_b:.4f}  (val_calib: {best_b})")

print("\n[방법 B] test 롤링 추론 & 평가...")
df_test_pred_b = predict_rolling(models_b, df_test, FEATURE_COLS)
test_disks_b = prepare_eval_data(df_test_pred_b)

rows_b = []
for thr in sorted({0.1, 0.5, 0.9, 0.99, round(thr_b, 4)}):
    s, _ = disk_level_eval_fast(test_disks_b, thr, LEAD_TIMES)
    rows_b.append(s)

df_res_b = pd.DataFrame(rows_b)
print(df_res_b.to_string(index=False))

[방법 B] val_calib으로 임계값 탐색 (FAR≤1%, lead=30d)...


  예측 확률 분포 백분위수:
count    7.975356e+06
mean     4.949928e-02
std      8.135300e-02
min      3.219296e-11
50%      3.133469e-02
90%      8.304070e-02
99%      5.033344e-01
99.9%    8.943595e-01
max      9.983699e-01
Name: pred_prob, dtype: float64
  평가 데이터 그룹화 및 정렬 중...


  최적 임계값: 0.8328  (val_calib: {'threshold': np.float64(0.8327662662662664), 'Disk_FAR(%)': np.float64(0.4939), 'Recall_30d(%)': np.float64(35.9788)})

[방법 B] test 롤링 추론 & 평가...


  평가 데이터 그룹화 및 정렬 중...


 threshold  Disk_FAR(%)  Recall_10d(%)  Recall_20d(%)  Recall_30d(%)
    0.1000      49.7249        14.6514        17.9170        21.4475
    0.5000       3.7003        24.8897        31.3327        34.5102
    0.8328       0.7437        25.5075        31.0680        33.7158
    0.9000       0.4172        25.1545        28.9497        30.0088
    0.9900       0.0000         1.8535         1.9417         1.9417


---
## 최종 비교 (최적 임계값 기준)

In [9]:
print("=" * 65)
print("베이스라인 비교 (FAR≤1% 최적 임계값, test 셋, 고장당일 포함)")
print("=" * 65)

s_a, _ = disk_level_eval_fast(test_disks_a, thr_a, LEAD_TIMES)
s_b, _ = disk_level_eval_fast(test_disks_b, thr_b, LEAD_TIMES)

cmp = pd.DataFrame([s_a, s_b], index=["방법A (No-Tricks)", "방법B (UnderBagging)"])
print(cmp.to_string())

print("\n[리드타임별 개선량]")
for lt in LEAD_TIMES:
    col  = f"Recall_{lt}d(%)"
    diff = s_b[col] - s_a[col]
    print(f"  Recall_{lt}d: A={s_a[col]:.2f}%  B={s_b[col]:.2f}%  (B-A={diff:+.2f}%)")

베이스라인 비교 (FAR≤1% 최적 임계값, test 셋, 고장당일 포함)
                    threshold  Disk_FAR(%)  Recall_10d(%)  Recall_20d(%)  Recall_30d(%)
방법A (No-Tricks)      0.262310       1.0702        42.2771        45.1898        46.3372
방법B (UnderBagging)   0.832766       0.7437        25.5075        31.0680        33.7158

[리드타임별 개선량]
  Recall_10d: A=42.28%  B=25.51%  (B-A=-16.77%)
  Recall_20d: A=45.19%  B=31.07%  (B-A=-14.12%)
  Recall_30d: A=46.34%  B=33.72%  (B-A=-12.62%)


In [10]:
# 결과 및 모델 저장
out = {
    "feature_cols": FEATURE_COLS,
    "method_A": {"threshold": thr_a, "metrics": s_a},
    "method_B": {"threshold": thr_b, "metrics": s_b},
}
with open(MODEL_DIR / "baseline_results.json", "w", encoding="utf-8") as f:
    json.dump(out, f, ensure_ascii=False, indent=2)

joblib.dump(model_a, MODEL_DIR / "model_a.pkl")
for i, m in enumerate(models_b):
    joblib.dump(m, MODEL_DIR / f"model_b_subset_{i:02d}.pkl")

print(f"✅ 저장 완료 → {MODEL_DIR}")

✅ 저장 완료 → C:\Workspace\06_ML_projdect\26_1_COIN\models\10_baseline
